# 02 — Main Analysis

Estimates of the effect of ratifying the Istanbul Convention on lethal violence
against women.

Only the methods permitted for each outcome in notebook 01 are estimated here.
The notebook is organised around the argument rather than around the estimators:
a primary outcome, a falsification test on an outcome the Convention does not
target, a sex-differenced outcome that removes shocks common to both sexes, and
the homicide category closest to the Convention's substantive subject.

Order of precedence for the thesis:

1. **Callaway–Sant'Anna** and **stacked DiD** — the headline estimates. Both are
   robust to the heterogeneous-timing bias that affects two-way fixed effects
   under staggered adoption.
2. **Two-way fixed effects** — reported as a benchmark, with its known bias
   stated.
3. **Event studies** — identification diagnostics. The shape of the path is
   informative; individual coefficients are not effect estimates.
4. **Negative control and triple difference** — do the estimates behave like a
   gender-specific effect, or like a change in homicide or in recording that
   affects both sexes?

**Input** `data/gbv_panel_analysis.csv` and
`outputs/diagnostics/outcome_method_permissions.csv`.
**Output** `outputs/tables/`, two figures in `outputs/figures/`.

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "src").exists():
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR / "src"))

from analysis_helpers import *          # noqa: F401,F403
import estimators as E
import scm as SCM

DATA_PATH = resolve_data_path(PROJECT_DIR)
OUT = make_output_dirs(PROJECT_DIR)
_d = load_and_prepare_data(DATA_PATH, OUT)
df, SAMPLES = _d["df"], _d["SAMPLES"]
country_info, inventory = _d["country_info"], _d["inventory"]
ALL_CTRL = _d["ALL_CTRL"]

FEAS = pd.read_csv(OUT["diagnostics"] / "outcome_method_permissions.csv")
print("\nMethods cleared by notebook 02:")
print(FEAS[["label","methods_allowed"]].to_string(index=False))

## 1. Two-way fixed effects, with the negative control alongside

The specification regresses the outcome on a treatment indicator with country
and year fixed effects, clustering standard errors on country.

The negative control is what makes the result interpretable. The Convention
addresses violence against women. If ratification moves male homicide by as much
as female homicide, the coefficient is capturing something common to both — a
general trend in homicide, or a change in how homicide is recorded — rather than
a gender-specific effect.

In [ ]:
rows = []
for k in ["fhr", "male_hom", "log_ratio", "fem_ipf"]:
    if not supports(k, "twfe"):
        continue
    r = E.feols(SAMPLES[k], k, ["did_interaction"], ["country","year"],
                cluster="country", name=f"TWFE {k}")
    row = E.coef_row(r, "did_interaction", OUTCOME_BY_KEY[k]["label"])
    row["outcome"] = k
    row["role"] = OUTCOME_BY_KEY[k]["role"]
    rows.append(row)
twfe_tab = export_table(rows, OUT["tables"] / "main_twfe_estimates.csv",
                        "TWFE main")
print(twfe_tab[["label","role","b","se","p","ci_low","ci_high","N"]].to_string(index=False))

## 2. Estimators robust to staggered adoption

With treatment adopted at different times, the two-way fixed effects coefficient
is a weighted average of many two-group comparisons, some of which use
already-treated countries as controls and can enter with negative weight. Two
estimators avoid this.

**Stacked DiD** builds a separate dataset for each adoption cohort, containing
that cohort and its clean controls over a common event window, then estimates a
single coefficient across the stacks.

**Callaway–Sant'Anna** estimates a separate ATT for each cohort and calendar
year against a stated comparison group, then aggregates. Both never-treated and
not-yet-treated comparison groups are reported, since with only seven untreated
countries the choice matters.

In [ ]:
rows = []
for k in ["fhr", "male_hom", "log_ratio"]:
    out, _ = E.stacked_did(SAMPLES[k], k)
    out["outcome"] = k; out["estimator"] = "Stacked DiD"
    out["label"] = OUTCOME_BY_KEY[k]["label"]
    rows.append(out)
stk = export_table(rows, OUT["tables"] / "main_stacked_did_estimates.csv", "stacked DiD")
print("STACKED DiD (cohort stacks; fixed effects absorbed; clustered on country)")
print(stk[["label","b","se","p","N","clusters","n_stacks"]].to_string(index=False))

In [ ]:
cs_rows = []
for k in ["fhr", "male_hom", "log_ratio"]:
    for cg in ["notyettreated", "nevertreated"]:
        r = E.callaway_santanna(SAMPLES[k], k, control_group=cg, n_boot=N_BOOT, seed=SEED)
        cs_rows.append({"label": OUTCOME_BY_KEY[k]["label"], "outcome": k,
                        "control_group": cg, "b": r["att_simple"], "se": r["se"],
                        "p": r["p"], "ci_low": r["ci_low"], "ci_high": r["ci_high"],
                        "n_boot": r["n_boot_used"]})
        if k == "fhr" and cg == "notyettreated":
            r["dynamic"].to_csv(OUT["tables"] / "callaway_santanna_dynamic_female_homicide.csv", index=False)
            r["attgt"].to_csv(OUT["tables"] / "callaway_santanna_attgt_female_homicide.csv", index=False)
            r["group"].to_csv(OUT["tables"] / "callaway_santanna_by_cohort_female_homicide.csv", index=False)
cs = export_table(cs_rows, OUT["tables"] / "main_callaway_santanna_estimates.csv", "Callaway-Sant'Anna")
print("CALLAWAY & SANT'ANNA (own implementation — see estimators.py for why the")
print("installed csdid package is not used: it ignores control_group)")
print(cs.to_string(index=False))

In [ ]:
# Cohort-level heterogeneity: the substantive explanation for the aggregate null.
grp = pd.read_csv(OUT["tables"] / "callaway_santanna_by_cohort_female_homicide.csv")
print("Group (cohort) specific ATTs, female homicide:")
print(grp.to_string(index=False))
print(f"\nRange: {grp.att.min():+.4f} to {grp.att.max():+.4f}. An aggregate average")
print("over effects this heterogeneous is not a useful summary on its own.")

## 3. Inference that does not rely on cluster-robust asymptotics

Cluster-robust standard errors are justified by an argument that assumes many
clusters. There are seven control countries. Two alternatives are reported: a
wild cluster bootstrap with the null imposed, and a permutation test that
reassigns the observed pattern of adoption timing across countries.

In [ ]:
inf_rows = []
for k in ["fhr", "male_hom", "log_ratio"]:
    w = E.wild_cluster_bootstrap(SAMPLES[k], k, ["did_interaction"], ["country","year"],
                                 cluster="country", test_var="did_interaction",
                                 n_boot=999, seed=SEED)
    p = E.permutation_test(SAMPLES[k], k, n_perm=N_PERM, seed=SEED)
    inf_rows.append({"label": OUTCOME_BY_KEY[k]["label"], "coef": w["coef"],
                     "p_cluster_robust": w["p_cluster"], "p_wild_bootstrap": w["p_wild"],
                     "p_permutation": p["p_permutation"], "n_perm": p["n_perm_used"]})
inf = pd.DataFrame(inf_rows)
inf.to_csv(OUT["tables"] / "inference_alternative_methods.csv", index=False)
print(inf.to_string(index=False))
print("\nPermutation inference reassigns the observed cohort structure across")
print("countries. It assumes ratification timing is exchangeable across countries,")
print("which is questionable — ratification correlates with country characteristics.")
print("Report all three p-values; do not pick the smallest.")

## 4. Event studies

Reported to show the shape of the path around ratification, not to read effects
off individual coefficients, which are imprecise. Endpoints are binned so that
treated observations outside the plotted window are retained rather than
dropped.

The figure shows female homicide, male homicide and their ratio on a common
scale. This is the central piece of evidence in the thesis.

In [ ]:
ES = {}
for k in ["fhr", "male_hom", "log_ratio", "fem_ipf", "svr"]:
    if not supports(k, "event_study"):
        continue
    es, diag, _ = E.event_study(SAMPLES[k], k)
    ES[k] = es
    es.to_csv(OUT["tables"] / f"event_study_{OUTCOME_BY_KEY[k]['file_slug']}.csv", index=False)
    print(f"\n[{OUTCOME_BY_KEY[k]['label']}]  n={diag['n']}  clusters={diag['n_clusters']}")
    print(es[["event_time","coef","se","p","n_treated_countries"]].to_string(index=False))

In [ ]:
# Female and male homicide are plotted on a SHARED y-axis: both are rates per
# 100,000, and the comparison between them is the point of the figure, so
# independent scales would misrepresent their relative magnitudes. The ratio is
# in log points and necessarily has its own axis.
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5), sharex=True)
lo = min(ES[k].ci_low.min() for k in ["fhr", "male_hom"])
hi = max(ES[k].ci_high.max() for k in ["fhr", "male_hom"])
pad = 0.05 * (hi - lo)

for ax, k in zip(axes, ["fhr", "male_hom", "log_ratio"]):
    es = ES[k].sort_values("event_time")
    ax.fill_between(es.event_time, es.ci_low, es.ci_high,
                    color=OUTCOME_BY_KEY[k]["color"], alpha=.20)
    ax.plot(es.event_time, es.coef, "o-", color=OUTCOME_BY_KEY[k]["color"],
            linewidth=2.2, markersize=6)
    ax.axhline(0, color="black", linestyle="--", linewidth=.9, alpha=.6)
    ax.axvline(-0.5, color=C_POLICY, linestyle="--", linewidth=1.4, alpha=.8)
    ax.set_title(OUTCOME_BY_KEY[k]["label"], fontsize=12)
    ax.set_xlabel("Years relative to ratification")
    ax.grid(alpha=.25)
    if k in ("fhr", "male_hom"):
        ax.set_ylim(lo - pad, hi + pad)

axes[0].set_ylabel("Effect (deaths per 100,000 population)")
axes[1].set_ylabel("Effect (deaths per 100,000 population)")
axes[2].set_ylabel("Effect (log points)")
fig.suptitle("Ratification moves male homicide at least as much as female homicide,\n"
             "and their ratio not at all", y=1.06, fontsize=15)
fig_note("Endpoints binned; reference period t=-1; cluster-robust 95% confidence "
         "intervals. The first two panels share a y-axis. Estimates are "
         "differences relative to never-ratifying countries, not causal effects "
         "unless parallel trends hold. Source: UNODC UN-CTS.")
save_fig(OUT["figures"], "event_study_gender_comparison.png")

## 5. Sensitivity to violations of parallel trends

Parallel trends cannot be tested, only probed. Following Rambachan and Roth
(2023), this asks how large a post-treatment differential trend would have to
be, relative to the largest differential movement already visible before
treatment, before the conclusion changes.

In [ ]:
bounds, meta = E.honest_rm_bounds(ES["fhr"], m_grid=(0.0, 0.5, 1.0, 2.0), post_max=8)
print("max observed pre-period step:", round(meta["max_pre_step"], 4))
print(meta["note"])
piv = bounds.pivot_table(index="event_time", columns="M",
                         values=["bound_low","bound_high"])
print(piv.round(3).to_string())
bounds.to_csv(OUT["tables"] / "parallel_trends_sensitivity_bounds.csv", index=False)
print("\nThe identified set already contains zero at M=0 (exact parallel trends),")
print("so no assumption about trend violations can rescue a signal that is not")
print("there. The bounds instead show how wide the range of effects consistent")
print("with the data is.")

## 6. Alternative control groups

Each row answers a different, stated question about who serves as the
counterfactual, and all rows are reported. The post-socialist restriction is the
most substantive: it compares post-socialist ratifiers with post-socialist
non-ratifiers, removing the regional imbalance in the main comparison at the
cost of a smaller sample and a narrower estimand.

This is not a search for a specification that produces significance. The point
of reporting all of them is to show how much the estimate depends on a choice
that theory does not pin down.

In [ ]:
alts = []
base = SAMPLES["fhr"]
specs = [
    ("All never-treated controls (7)", base),
    ("Excluding Lithuania", base[base.country != "Lithuania"]),
    ("Czechia/Hungary/Slovakia controls only",
     base[(base.treated_ever==1) | base.country.isin(["Czechia","Hungary","Slovakia"])]),
    ("Post-socialist countries only (treated AND control)",
     comparability_restricted(base, "post_socialist")),
]
for label, s in specs:
    if s.empty or s.did_interaction.nunique() < 2:
        continue
    r = E.feols(s, "fhr", ["did_interaction"], ["country","year"],
                cluster="country", name=label)
    row = E.coef_row(r, "did_interaction", label)
    row["treated_countries"] = int(s.loc[s.treated_ever.eq(1),"country"].nunique())
    row["control_countries"] = int(s.loc[s.treated_ever.eq(0),"country"].nunique())
    alts.append(row)
alt = export_table(alts, OUT["tables"] / "control_group_sensitivity.csv", "control alternatives")
print(alt[["label","b","se","p","N","treated_countries","control_countries"]].to_string(index=False))
print("\nThe post-socialist restriction is the only one that removes the")
print("treated/control composition imbalance rather than just trimming outliers.")
print("It estimates a DIFFERENT estimand: the ATT among post-socialist ratifiers.")

## 7. Consolidated results and multiple testing

`outputs/tables/main_results.csv` is the authoritative table. Every coefficient
quoted in the thesis should come from it.

The Holm correction is applied to the confirmatory family only — the single
pre-specified test on female homicide. The negative control, the ratio and the
targeted outcome are reported without correction because they serve different
inferential roles rather than being repeated tests of the same hypothesis.

In [ ]:
master = []
for _, r in twfe_tab.iterrows():
    master.append({"outcome": r["outcome"], "label": r["label"], "estimator": "TWFE",
                   "b": r["b"], "se": r["se"], "p": r["p"], "N": r["N"]})
for _, r in stk.iterrows():
    master.append({"outcome": r["outcome"], "label": r["label"], "estimator": "Stacked DiD",
                   "b": r["b"], "se": r["se"], "p": r["p"], "N": r["N"]})
for _, r in cs[cs.control_group=="notyettreated"].iterrows():
    master.append({"outcome": r["outcome"], "label": r["label"],
                   "estimator": "Callaway-Sant'Anna (not-yet-treated)",
                   "b": r["b"], "se": r["se"], "p": r["p"], "N": np.nan})
mast = pd.DataFrame(master)

# Holm applies to the confirmatory family only. Applying it across the
# five-outcome family included outcomes that should never have been tested.
conf = mast[(mast.outcome=="fhr")]
conf = conf.assign(p_holm=holm(conf.p.values))
print("Confirmatory family (female homicide, one outcome, three estimators):")
print(conf[["estimator","b","se","p","p_holm"]].to_string(index=False))
mast.to_csv(OUT["tables"] / "main_results.csv", index=False)
print("\nAll estimators, all outcomes:")
print(mast.to_string(index=False))

In [ ]:
# Two panels, because the outcomes are in different units. The two homicide
# rates share an axis in deaths per 100,000; the ratio is in log points and
# cannot be plotted against them.
fig, (ax_rate, ax_ratio) = plt.subplots(
    1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [2, 1]})

ORDER = ["TWFE", "Stacked DiD", "Callaway-Sant'Anna (not-yet-treated)"]
colors = {"fhr": C_TREATED, "male_hom": C_NEG, "log_ratio": C_MID}

rates = mast[mast.outcome.isin(["fhr", "male_hom"])].copy()
rates["ypos"] = range(len(rates))
for _, r in rates.iterrows():
    ax_rate.errorbar(r.b, r.ypos, xerr=1.96 * r.se, fmt="o", capsize=4,
                     markersize=8, color=colors[r.outcome], linewidth=2)
ax_rate.set_yticks(rates.ypos)
ax_rate.set_yticklabels(
    [f"{r.label.replace(' (negative control)', '')} - {r.estimator}"
     for _, r in rates.iterrows()], fontsize=9)
ax_rate.set_xlabel("Estimate, deaths per 100,000 (95% CI)")
ax_rate.set_title("Female homicide (blue) and male homicide (teal)", fontsize=12)

ratio = mast[mast.outcome.eq("log_ratio")].copy()
ratio["ypos"] = range(len(ratio))
for _, r in ratio.iterrows():
    ax_ratio.errorbar(r.b, r.ypos, xerr=1.96 * r.se, fmt="o", capsize=4,
                      markersize=8, color=colors["log_ratio"], linewidth=2)
ax_ratio.set_yticks(ratio.ypos)
ax_ratio.set_yticklabels([r.estimator for _, r in ratio.iterrows()], fontsize=9)
ax_ratio.set_xlabel("Estimate, log points (95% CI)")
ax_ratio.set_title("log(female / male homicide)", fontsize=12)

for ax in (ax_rate, ax_ratio):
    ax.axvline(0, color="black", linestyle="--", linewidth=1, alpha=.6)
    ax.grid(True, axis="x", alpha=.25)

fig.suptitle("Main estimates by outcome and estimator", y=1.02, fontsize=15)
fig_note("Estimates are differences relative to countries that had not ratified "
         "by 2023. Panels use different units and are not on a common scale. "
         "Source: UNODC UN-CTS.")
save_fig(OUT["figures"], "main_estimates_by_method.png")
print("\nMain analysis complete.")